# KLEOS 00 — Environment checkRun this first on any new Colab runtime. It answers two questions before youspend a session on anything expensive:1. Is the environment set up correctly?2. **Which models can this particular runtime actually train?**Free-tier Colab assigns different GPUs at different times. A T4 (16GB) trains an8B model in 4-bit comfortably and cannot train a 24B model at all. Finding thatout here takes seconds; finding it out after a 28GB download does not.

## 1. Clone the repository

In [ ]:
# Clone the repository (skip if already present) and enter it.import osfrom pathlib import PathREPO_DIR = Path("/content/kleos-models")if not REPO_DIR.exists():    !git clone https://github.com/kleos/kleos-models.git {REPO_DIR}else:    print(f"{REPO_DIR} already exists; pulling latest")    !cd {REPO_DIR} && git pull --ff-onlyos.chdir(REPO_DIR)print("working directory:", Path.cwd())

## 2. Install dependencies

In [ ]:
# Install dependencies WITHOUT touching Colab's torch build.## Reinstalling torch on Colab replaces the build compiled against this runtime's# CUDA driver, and CUDA then silently stops working. scripts/colab_setup.py uses# --no-deps for every package that would otherwise pull torch along.!python scripts/colab_setup.py

## 3. Check the assigned GPUThe key numbers are total VRAM and compute capability. Compute capability below8.0 (a T4 is 7.5) means no bfloat16, so training uses float16 instead.

In [ ]:
# What GPU did Colab actually assign? Free-tier allocation varies.!nvidia-smifrom kleos_models.models.feasibility import probe_gpugpu = probe_gpu()print()print(gpu.render())print()if not gpu.available:    print("NO GPU ASSIGNED.")    print("Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")elif not gpu.bf16_supported:    print(f"Note: {gpu.name} (compute capability {gpu.capability_string}) has no")    print("bfloat16 support. KLEOS configs use compute_dtype: auto, which selects")    print("float16 here automatically. Nothing to change.")

## 4. Authenticate (optional)

In [ ]:
# Hugging Face authentication.## Needed only for gated base models (Mistral) or to publish an adapter.# Use Colab Secrets (the key icon in the left sidebar), never a literal token in# a cell — notebooks get shared and committed.import ostry:    from google.colab import userdata    token = userdata.get("HF_TOKEN")    if token:        os.environ["HF_TOKEN"] = token        print("HF_TOKEN loaded from Colab secrets.")    else:        print("No HF_TOKEN secret set.")except Exception as exc:    print(f"Colab secrets unavailable ({type(exc).__name__}).")    print("Set os.environ['HF_TOKEN'] manually if you need gated models.")if not os.environ.get("HF_TOKEN"):    print()    print("Without a token you can still use ungated models such as Qwen/Qwen3-8B.")    print("To add one: sidebar key icon -> Add new secret -> name HF_TOKEN ->")    print("enable 'Notebook access'.")

## 5. Feasibility across every model configThis is the cell worth reading carefully. It estimates peak memory for eachsupported model **on the GPU you were actually assigned**, with no downloads.

In [ ]:
!python scripts/plan_run.py --all-models

### Reading the table| Tier | Meaning || --- | --- || `full_research` | Fits with headroom. Use it. || `adapter_train` | Fits, but tight. Evaluation spikes may still OOM. || `smoke` | Only a reduced config fits — the research config does not. || `inference_only` | Can be evaluated here, not trained. || `infeasible` | Cannot even be loaded. |On a free T4 expect `qwen3_8b` and `ministral_8b` to be trainable, and`mistral_small_3_2` and `qwen3_30b_a3b_thinking` to be infeasible. That is not abug — those need an A100-class GPU. The estimates are deliberately slightlypessimistic.

## 6. Smoke test

In [ ]:
!python scripts/smoke_test.py

Everything above should pass or be explicitly skipped. A skip means an optionaldependency is missing; a failure means something is wrong and training will notwork.## Next- `01_dataset_validation.ipynb` — check your dataset before training on it- `02_train_qlora.ipynb` — the canonical training notebook